In [1]:
import sys,os
import imagej
import scyjava as sj
import tifffile as tff
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import cv2
import re
import pandas as pd

In [2]:
import os
import pandas as pd
os.chdir("/Users/cybellesmith/Box Sync/all/post_doc2/code/kiera_data_analysis/")
well_time_table = pd.read_csv("well_time_table_updated.csv")
print(well_time_table)

   well  patt1_start_min  patt1_end_min  patt2_start_min  patt2_end_min  \
0     3              2.5      17.500000        18.500000      33.500000   
1     4              2.0      17.000000        18.000000      33.000000   
2     5              2.0      17.000000        18.000000      33.000000   
3     7              2.0      17.000000        18.000000      33.000000   
4     8              2.5      17.500000        18.500000      33.500000   
5     9              2.0      17.000000        18.000000      33.000000   
6    10              3.0      16.499665        17.749665      32.749665   
7    11              2.0      17.000000        18.000000      33.000000   

                                               notes  
0                                                NaN  
1                                                NaN  
2                                                NaN  
3                                                NaN  
4                                                

In [ ]:
#load ND2 and convert to tif, all files

# Set the path to ImageJ/Fiji
IMAGEJ_PATH = "/Applications/Fiji.app"  # Adjust if necessary

# start up an imagej session
ij = imagej.init(IMAGEJ_PATH, headless=True)
# get bioformats importer
BF = sj.jimport("loci.plugins.BF")
options = sj.jimport("loci.plugins.in.ImporterOptions")() # import and initialize ImporterOptions
options.setVirtual(True) # load virtually to avoid overloading RAM

filenames_tif = []

for video_file in filenames:

    video_full_path = nd2_video_dir / video_file
    options.setId(str(video_full_path)) #this sets the input file
    
    # open the series
    vs = BF.openImagePlus(options)
    img = vs[0]
    
    # get dimensions of the series
    video_dims = ij.py.from_java(img.getDimensions())
    ntimepts = video_dims[4]
    
    video = []
    
    #load in every frame of the video
    for slice_idx in range(1, ntimepts + 1):
        if slice_idx % 100 == 0:
            print(f"Processing slice {slice_idx}/{ntimepts} for {video_file}...")
        img.setSlice(slice_idx)
        frame_np = ij.py.from_java(img.getProcessor().getPixels()) # returns a vector of pixel values at one timepoint
        frame_np = frame_np.reshape((video_dims[1], video_dims[0])) # reshapes as a 2D frame
        video.append(frame_np)
    
    video = np.array(video)

    os.chdir(output_dir)

    video_file_tif = re.sub(r'\.nd2$', '.tif', video_file)

    filenames_tif.append(video_file_tif)
    
    tff.imwrite(video_file_tif, video)


In [ ]:
# define helper functions for applying motion correction (given method)

import numpy as np
import cv2
import time

# ---------------------------------------------------------------------
# Utilities

def _norm01(x):
    x = x.astype(np.float32)
    mn, mx = x.min(), x.max()
    if mx > mn: x = (x - mn) / (mx - mn)
    else:       x = np.zeros_like(x, dtype=np.float32)
    return x

def _to_gray32(x):
    x = x.astype(np.float32)
    if x.ndim == 3:
        x = cv2.cvtColor(x, cv2.COLOR_BGR2GRAY)
    return x

def _get_reference(frames, reference="middle"):
    if reference == "median": return np.median(frames, axis=0).astype(frames.dtype)
    if reference == "first":  return frames[0]
    if reference == "last":   return frames[-1]
    return frames[len(frames)//2]

def safe_phase_correlate(ref, mov, window=None):
    """
    Robust wrapper for cv2.phaseCorrelate.
    Avoids hangs from NaNs, Infs, or blank frames.
    Returns ((dx, dy), resp).
    """
    ref32 = np.ascontiguousarray(ref.astype(np.float32))
    mov32 = np.ascontiguousarray(mov.astype(np.float32))

    if not np.isfinite(ref32).all() or not np.isfinite(mov32).all():
        # skip and warn
        print("⚠️ NaN/Inf detected in phaseCorrelate input — skipping frame.")
        return (0.0, 0.0), 0.0

    if np.std(ref32) < 1e-6 or np.std(mov32) < 1e-6:
        # blank / constant frames
        return (0.0, 0.0), 0.0

    try:
        (dx, dy), resp = cv2.phaseCorrelate(ref32, mov32, window)
        return (dx, dy), float(resp)
    except cv2.error as e:
        print(f"⚠️ cv2.phaseCorrelate failed: {e}")
        return (0.0, 0.0), 0.0

# ---------------------------------------------------------------------
# Aligners
def align_translation_phasecorr(ref, img):
    """Pure translation via phase correlation (fast)."""
    ref32 = _norm01(_to_gray32(ref))
    img32 = _norm01(_to_gray32(img))
    (dx, dy), _ = safe_phase_correlate(ref32, img32)
    M = np.array([[1, 0, -dx],[0, 1, -dy]], dtype=np.float32)
    aligned = cv2.warpAffine(img, M, (ref.shape[1], ref.shape[0]),
                             flags=cv2.INTER_LINEAR,
                             borderMode=cv2.BORDER_REFLECT)
    return aligned, M

def align_ecc_model(ref, img, motion=cv2.MOTION_EUCLIDEAN,
                    max_iter=80, eps=1e-6, gaussFiltSize=5,
                    init_from_phasecorr=True, scale=0.5):
    """
    ECC alignment with optional downsampling (scale<1). Transform is
    estimated at low-res and applied at full-res.
    """
    ref32 = _norm01(_to_gray32(ref))
    img32 = _norm01(_to_gray32(img))

    if scale != 1.0:
        fx = fy = float(scale)
        refS = cv2.resize(ref32, None, fx=fx, fy=fy, interpolation=cv2.INTER_AREA)
        imgS = cv2.resize(img32, None, fx=fx, fy=fy, interpolation=cv2.INTER_AREA)
    else:
        refS, imgS = ref32, img32

    W = (np.eye(3, dtype=np.float32) if motion == cv2.MOTION_HOMOGRAPHY
         else np.eye(2, 3, dtype=np.float32))

    if init_from_phasecorr and motion in (cv2.MOTION_TRANSLATION, cv2.MOTION_EUCLIDEAN, cv2.MOTION_AFFINE):
        (dxS, dyS), _ = safe_phase_correlate(refS, imgS)
        W[:2, 2] = (dxS, dyS)

    criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, int(max_iter), float(eps))

    if np.std(imgS) < 1e-3 or np.std(refS) < 1e-3:
        # Very low contrast: ECC will hang or diverge -- so use phase correlation instead
        aligned, M = align_translation_phasecorr(ref, img)
        return aligned, M, 0.0
    
    try:
        cc, W = cv2.findTransformECC(refS, imgS, W, motion, criteria, None, gaussFiltSize)
    except cv2.error:
        aligned, M = align_translation_phasecorr(ref, img)
        return aligned, M, 0.0

    # Upscale translation components for full-res application
    if scale != 1.0 and motion != cv2.MOTION_HOMOGRAPHY:
        s = float(scale)
        W = W.copy()
        W[0, 2] /= s
        W[1, 2] /= s

    if motion == cv2.MOTION_HOMOGRAPHY:
        aligned = cv2.warpPerspective(img, W, (ref.shape[1], ref.shape[0]),
                                      flags=cv2.INTER_LINEAR | cv2.WARP_INVERSE_MAP,
                                      borderMode=cv2.BORDER_REFLECT)
    else:
        aligned = cv2.warpAffine(img, W, (ref.shape[1], ref.shape[0]),
                                 flags=cv2.INTER_LINEAR | cv2.WARP_INVERSE_MAP,
                                 borderMode=cv2.BORDER_REFLECT)
    return aligned, W, float(cc)

def align_nonrigid_flow(ref, img, method="dis"):
    """
    Dense optical flow warping (nonrigid). Default to DIS (fast).
    """
    ref32 = _norm01(_to_gray32(ref))
    img32 = _norm01(_to_gray32(img))

    # Ensure contiguous arrays (some cv2 funcs care)
    ref32 = np.ascontiguousarray(ref32)
    img32 = np.ascontiguousarray(img32)

    if method == "dis":
        # DIS needs 8-bit single-channel input
        ref8 = (ref32 * 255.0).astype(np.uint8)
        img8 = (img32 * 255.0).astype(np.uint8)

        of = cv2.DISOpticalFlow_create(cv2.DISOPTICAL_FLOW_PRESET_MEDIUM)
        # You can tweak these if needed:
        # of.setPatchSize(8)
        # of.setFinestScale(1)
        flow = of.calc(ref8, img8, None)  # HxWx2
    else:
        # Farneback accepts 8U or 32F single-channel
        flow = cv2.calcOpticalFlowFarneback(
            ref32, img32, None,
            pyr_scale=0.5, levels=3, winsize=21, iterations=2,
            poly_n=5, poly_sigma=1.2, flags=0
        )

    h, w = ref32.shape
    grid_x, grid_y = np.meshgrid(np.arange(w), np.arange(h))
    map_x = (grid_x + flow[..., 0]).astype(np.float32)
    map_y = (grid_y + flow[..., 1]).astype(np.float32)

    if img.ndim == 3:
        chans = [cv2.remap(img[..., c], map_x, map_y,
                           interpolation=cv2.INTER_LINEAR,
                           borderMode=cv2.BORDER_REFLECT)
                 for c in range(img.shape[2])]
        aligned = np.stack(chans, axis=2)
    else:
        aligned = cv2.remap(img, map_x, map_y,
                            interpolation=cv2.INTER_LINEAR,
                            borderMode=cv2.BORDER_REFLECT)
    return aligned, flow

# ---------------------------------------------------------------------
# Stack dispatcher with speed knobs
def align_stack(frames,
                model='euclidean',          # 'translation' | 'translation_ref' | 'euclidean' | 'affine_ecc' | 'nonrigid_flow'
                reference='middle',
                flow_method='dis',
                ecc_scale=0.5,              # downsample factor for ECC
                ecc_max_iter=80,
                return_params=False):
    assert frames.ndim >= 3, "frames must be (T,H,W[,...])"
    T = frames.shape[0]
    ref = _get_reference(frames, reference=reference)
    out = np.empty_like(frames)
    params = []

    ecc_modes = {'translation': cv2.MOTION_TRANSLATION,
                 'euclidean':   cv2.MOTION_EUCLIDEAN,
                 'affine_ecc':  cv2.MOTION_AFFINE}

    ref32_cache = _norm01(_to_gray32(ref))  # reuse for translation

    for i in range(T):
        f = frames[i]
        if model in ('translation', 'translation_ref'):
            try:
                img32 = _norm01(_to_gray32(f))
                (dx, dy), _ = safe_phase_correlate(ref32_cache, img32)
                M = np.array([[1, 0, -dx],[0, 1, -dy]], dtype=np.float32)
                a = cv2.warpAffine(f, M, (ref.shape[1], ref.shape[0]),
                                   flags=cv2.INTER_LINEAR,
                                   borderMode=cv2.BORDER_REFLECT)
                out[i] = a; params.append(M)
            except cv2.error:
                out[i] = f; params.append(None)

        elif model in ('euclidean', 'affine_ecc'):
            motion = ecc_modes[model]
            a, W, cc = align_ecc_model(ref, f, motion=motion,
                                       max_iter=ecc_max_iter, scale=ecc_scale)
            out[i] = a; params.append((W, cc))

        elif model == 'nonrigid_flow':
            a, flow = align_nonrigid_flow(ref, f, method=flow_method)
            out[i] = a; params.append(flow)

        else:
            raise ValueError(f"Unknown model: {model}")

    return (out, params) if return_params else out

def motion_correct(image, model='euclidean', reference='middle', **kwargs):
    return align_stack(image, model=model, reference=reference, **kwargs)

In [ ]:
#motion correct all the videos -- first pass = nonrigid_flow

os.chdir(output_dir)

all_flow = []
video_files_mc = []

for video_file in filenames_tif:

    print(video_file)

    video = tff.imread(video_file)

    video_mc, flow = motion_correct(video, model='nonrigid_flow', reference="middle", return_params=True)

    all_flow.append(flow)

    video_file_mc = re.sub(r'\.tif$', '_mc.tif', video_file)

    video_files_mc.append(video_file_mc)

    tff.imwrite(video_file_mc,video_mc)

In [ ]:
#check all motion correction by hand/eye. Redo motion correction for select files as needed.

#well 5 looks fine
#well 4 looks fine
#well 3 looks fine (and also checked more thoroughly earlier)
#well 7 looks fine (larger FOV)
#well 8 looks fine
#well 11 -- A bit off -- one weird warp. But not too bad. Seems useable for our purposes (given event blobs can shift slightly in space...)
#well 10 t1 -- ugh weird warping due to rapid movement early in the time point frames 210-217 with 1-indexing... fuck.... 
            #also for this analysis we don't HAVE to align t1 and t2 but probably a good idea...
#well 10 t2 -- looks good.
#well 9 looks fine


In [ ]:

#well 10. --> do initial rigid-body alignment only to correct that one weird section, then do nonrigid flow, align and combine t1 and t2

#test align frame 210 to frame 216, rigid body

os.chdir(output_dir)

video_file = "DIV27_HS1-DA1M_W10_optostim_t1.tif"

video = tff.imread(video_file)

test = np.array([video[209,:,:], video[216,:,:]])

test_mc = motion_correct(test, model="translation", reference="last", return_params=False)

plt.imshow(test_mc[0,:,:])
plt.show()
plt.imshow(test_mc[1,:,:])
plt.show()

In [ ]:
def translation_mc_to_ref_img(video, ref, return_params=False):
    video = np.concatenate([video, ref])
    video_mc = motion_correct(video, model="translation", reference="last", return_params=False)
    video_mc = np.array([video_mc[i,:,:] for i in range(0,len(video_mc) - 1)])
    return(video_mc)

In [ ]:
part1 = np.array([video[i,:,:] for i in range(0,217)])
part2 = np.array([video[i,:,:] for i in range(217,len(video))])
print(len(video))
print(len(part1))
print(len(part2))

video_t2 = tff.imread("DIV27_HS1-DA1M_W10_optostim_t2.tif")
v2_first_frame = np.array([video_t2[0]])

part1_mc = translation_mc_to_ref_img(part1, v2_first_frame)

part1_mc_subset1 = np.array([part1_mc[i,:,:] for i in range(0,210)])
part1_mc_subset1_mc = motion_correct(part1_mc_subset1, model="nonrigid_flow", reference="middle", return_params=False)

part1_mc_subset2 = np.array([part1_mc[i,:,:] for i in range(210,217)])

part1_done = np.concatenate((part1_mc_subset1_mc, part1_mc_subset2), axis=0)

part2_mc  = translation_mc_to_ref_img(part2, v2_first_frame)

part2_mc = np.concatenate((part2_mc, video_t2), axis=0)

part2_done = motion_correct(part2_mc, model="nonrigid_flow", reference="middle", return_params=False)

video_mc = np.concatenate((part1_done, part2_done), axis=0)

print(len(video_mc))

video_file_mc = "DIV27_HS1-DA1M_W10_optostim_mc.tif"
os.chdir(output_dir)
tff.imwrite(video_file_mc,video_mc)

In [ ]:

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
pd.set_option('display.max_columns', None)

print(well_time_table.loc[well_time_table["well"] == 10,"notes"])

In [ ]:
#also, need to correct the table and timestamps for well 10...

pattern = re.compile(r"DIV27_HS1-DA1M_W10_optostim_t1.nd2")
well10_part1_idx = [i for i, s in enumerate(filenames) if pattern.search(s)][0]
print(well10_part1_idx)
patt1_end_time_ms = all_timestamps[well10_part1_idx][-1]
print(patt1_end_time_ms)
patt1_end_time_min = patt1_end_time_ms / 1000.0 / 60
print(patt1_end_time_min)

well_time_table.loc[well_time_table["well"] == 10,"patt1_end_min"] = patt1_end_time_min
well_time_table.loc[well_time_table["well"] == 10,"patt2_start_min"] = patt1_end_time_min + well_time_table.loc[well_time_table["well"] == 10,"patt2_start_min"]
well_time_table.loc[well_time_table["well"] == 10,"patt2_end_min"] = patt1_end_time_min + well_time_table.loc[well_time_table["well"] == 10,"patt2_end_min"]


In [ ]:
os.chdir("/Users/cybellesmith/Box Sync/all/post_doc2/code/kiera_data_analysis/")
well_time_table.to_csv('well_time_table_updated.csv', index=False)
print(well_time_table)

In [ ]:
#well 11 -- correct weird warping...
#try doing a translational alignment pass before the nonrigid flow pass...

os.chdir(output_dir)

video_file = "DIV27_HS1-DA1M_W11_optostim.tif"

video = tff.imread(video_file)

video_mc1 = motion_correct(video, model="translation", reference="middle", return_params=False)

video_mc2 = motion_correct(video_mc1, model="nonrigid_flow", reference="middle", return_params=False)

video_file_mc = "DIV27_HS1-DA1M_W11_optostim_mc1.tif"
os.chdir(output_dir)
tff.imwrite(video_file_mc,video_mc1)

video_file_mc = "DIV27_HS1-DA1M_W11_optostim_mc2.tif"
os.chdir(output_dir)
tff.imwrite(video_file_mc,video_mc2)


In [ ]:
frame_to_switch_mc = 2600
part1 = np.array([video_mc2[i,:,:] for i in range(0,frame_to_switch_mc)])
part2 = np.array([video_mc1[i,:,:] for i in range(frame_to_switch_mc, len(video))])

part1_last_frame = np.array([part1[-1]])
part2 = translation_mc_to_ref_img(part2, part1_last_frame)

video_mc = np.concatenate((part1, part2), axis=0)

video_file_mc = "DIV27_HS1-DA1M_W11_optostim_mc3.tif"
os.chdir(output_dir)
tff.imwrite(video_file_mc,video_mc)

In [ ]:
#by hand -- all the mc files to use were changed to end with just "mc"

In [3]:
video_files_mc = [
'DIV27_CTRL-CTRL_W5_optostim_mc.tif',
 'DIV27_CTRL-CTRL_W4_optostim_mc.tif',
 'DIV27_CTRL-CTRL_W3_optostim_mc.tif',
 'DIV27_HS1-DA1M_W7_optostim_mc.tif',
 'DIV27_HS1-DA1M_W8_optostim_mc.tif',
 'DIV27_HS1-DA1M_W11_optostim_mc.tif',
 'DIV27_HS1-DA1M_W10_optostim_mc.tif',
 'DIV27_HS1-DA1M_W9_optostim_mc.tif'  
]

In [ ]:
#get all the max projection images from the motion corrected files

all_image_max_proj = []

for video in all_videos:
    #get max projection image of difference from mean trendline
    image_max_proj = np.max(video,axis=0)
    all_image_max_proj.append(image_max_proj)
    plt.imshow(image_max_proj, cmap='gray')
    plt.show()

In [ ]:
for video in all_videos:
    plt.hist(video.flatten())
    plt.show()

In [ ]:
from scipy.ndimage import gaussian_filter

sigma = 0.8
percent_exclude = 70

initial_masks = []

for image_max_proj in all_image_max_proj:
    image_max_proj_lp = gaussian_filter(image_max_proj, sigma=sigma)
    plt.imshow(image_max_proj)
    plt.show()
    plt.imshow(image_max_proj_lp)
    plt.show()
    #threshold top 10% of smoothed pixels
    thresh = np.percentile(image_max_proj_lp.flatten(), percent_exclude)
    init_mask = image_max_proj_lp > thresh
    plt.imshow(init_mask)
    plt.show()
    initial_masks.append(init_mask)
    

In [ ]:
import cv2
import numpy as np

def flood_fill_black_to_white(img_gray, seed_xy, lo=0, hi=0, connectivity=4):
    """
    img_gray: (H,W) uint8 or uint16 or float image (grayscale)
    seed_xy: (x, y) seed point (OpenCV uses x,y)
    lo/hi: tolerance below/above seed value (use >0 for "near black")
    connectivity: 4 or 8
    """
    img = img_gray.copy()

    # OpenCV wants uint8 for floodFill most reliably
    if img.dtype != np.uint8:
        # rescale to 0..255 if needed
        im = img.astype(np.float32)
        im -= im.min()
        im /= (im.max() - im.min() + 1e-6)
        img = (im * 255).astype(np.uint8)

    h, w = img.shape[:2]
    mask = np.zeros((h + 2, w + 2), dtype=np.uint8)  # must be 2 pixels larger

    flags = connectivity | cv2.FLOODFILL_FIXED_RANGE
    new_val = 255  # white

    # flood fill
    cv2.floodFill(img, mask, seedPoint=seed_xy, newVal=new_val,
                  loDiff=lo, upDiff=hi, flags=flags)

    return img

# example usage:
# seed_xy = (511,128)
# filled = flood_fill_black_to_white(canvas_gray, seed_xy=seed_xy, lo=5, hi=5, connectivity=8)


In [ ]:
# Kernel size controls how big holes get filled

candidate_masks = []

k = 7
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))

k_big = 15   # try 3, 5, 7, 11 depending on hole size
kernel_big = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_big, k_big))

for i, mask in enumerate(initial_masks):
    
    image = mask.astype(np.uint8) * 255
    plt.imshow(image)
    plt.show()

    if i == 4: #close gap to image right edge befor flood fill for only one image
    
        image = cv2.morphologyEx(image, cv2.MORPH_CLOSE, kernel_big)
        
        plt.imshow(image)
        plt.show()
    
    seed_xy = (image.shape[1]-1,int(np.round(image.shape[0]/2)))
    filled = flood_fill_black_to_white(image, seed_xy=seed_xy, lo=5, hi=5, connectivity=8)

    plt.imshow(filled)
    plt.show()

    filled = cv2.morphologyEx(filled, cv2.MORPH_CLOSE, kernel)
    
    plt.imshow(filled)
    plt.show()    

    candidate_masks.append(filled)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def draw_mask_contours_red(mask, img, thickness=2):
    """
    mask: 2D boolean/0-1/0-255 array (H,W)
    img:  2D grayscale (H,W) or color (H,W,3) image to draw on
    returns: color image with red contour lines
    """
    # --- ensure mask is uint8 0/255 ---
    mask_u8 = (mask > 0).astype(np.uint8) * 255

    # --- find contours (external only; use RETR_TREE if you want holes too) ---
    contours, hierarchy = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # --- ensure we have a BGR canvas to draw on ---
    if img.ndim == 2:
        canvas = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    else:
        canvas = img.copy()

    # --- draw red contours (BGR: (0,0,255)) ---
    cv2.drawContours(canvas, contours, contourIdx=-1, color=(0, 0, 255), thickness=thickness)

    return canvas, contours, hierarchy

# Example usage:
# mask: (H,W) boolean or 0/1 array
# base_img: (H,W) or (H,W,3)
# overlay, contours, hierarchy = draw_mask_contours_red(mask, base_img, thickness=2)

# plt.figure(figsize=(7,7))
# plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
# plt.axis("off")
# plt.title("Mask contours (red lines)")
# plt.show()

In [8]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def to_uint8(img):
    """Normalize any numeric image to uint8 [0,255] for display/cv2 ops."""
    img = img.astype(np.float32)
    mn, mx = np.nanmin(img), np.nanmax(img)
    if mx <= mn:
        return np.zeros(img.shape, dtype=np.uint8)
    img = (img - mn) / (mx - mn)
    return (img * 255).clip(0, 255).astype(np.uint8)

In [ ]:
for i, mask in enumerate(candidate_masks):
    base = all_image_max_proj[i]

    # Convert unsupported int16 -> uint8 for cv2/matplotlib display
    base_u8 = to_uint8(base)

    # If it's BGR color, convert to RGB for matplotlib; if grayscale, leave it.
    if base_u8.ndim == 3 and base_u8.shape[2] == 3:
        image_rgb = cv2.cvtColor(base_u8, cv2.COLOR_BGR2RGB)
    else:
        image_rgb = base_u8  # grayscale OK

    overlay, contours, hierarchy = draw_mask_contours_red(mask > 0, image_rgb, thickness=2)

    plt.figure(figsize=(6,6))
    # overlay is BGR if draw_mask_contours_red produced BGR; convert for plotting if needed
    if overlay.ndim == 3 and overlay.shape[2] == 3:
        plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(overlay, cmap="gray")
    plt.axis("off")
    plt.show()

In [ ]:
#ok -- the microcolumn masks seem fine -- maybe not perfect but good enough!

masks = [mask > 0 for mask in candidate_masks]
mask_backgrounds = [mask == 0 for mask in candidate_masks]

In [ ]:
def get_dff(video,mask,mask_background):
    
    ntimesteps = video.shape[0]
    
    raw_pixel_signal = np.array([video[t][mask] for t in range(ntimesteps)])
    
    #extract background mean for each timestep
    fnaut = np.array([np.mean(video[t][mask_background]) for t in range(ntimesteps)])
    
    delta_f_over_fnaut = np.array([np.divide(raw_pixel_signal[t] - fnaut[t],fnaut[t]) for t in range(ntimesteps)])
    
    return(delta_f_over_fnaut)

In [ ]:
all_dff = []
for i, video in enumerate(all_videos):
    mask = masks[i]
    mask_background = mask_backgrounds[i]
    dff = get_dff(video,mask,mask_background)
    all_dff.append(dff)

In [ ]:
all_mean_dff = []
for i in range(len(all_videos)):
    mean_dff = np.mean(all_dff[i],axis=1)
    all_mean_dff.append(mean_dff)
    plt.plot(mean_dff)
    plt.show()

In [ ]:
all_dff_video = []
for i, video in enumerate(all_videos):
    new = np.zeros(video.shape, dtype=np.float32)
    mask = masks[i]
    for t in range(len(video)):
        new[t][mask] = all_dff[i][t]
    all_dff_video.append(new)

In [ ]:
os.chdir(output_dir)

for i, video in enumerate(all_dff_video):
    filename = re.sub(r'\.tif$', '_dff.tif', video_files_mc[i])
    tff.imwrite(filename,video)

In [7]:
#reload in videos (as needed)

output_dir = "/Users/cybellesmith/Box Sync/all/post_doc2/code/kiera_data_analysis/processed_videos"

os.chdir(output_dir)

all_dff = []

for video_filename_mc in video_files_mc:
    filename = re.sub(r'\.tif$', '_dff.tif', video_filename_mc)
    video = tff.imread(filename)
    all_dff.append(video)

In [8]:
#ok -- but I still see neural activity in here -- more aggressive blurring + low pass filtering

#step 1 = save the spatially blurred videos

from scipy.ndimage import gaussian_filter

os.chdir(output_dir)

sigma = 20

dff_blurred = []

for i, video in enumerate(all_dff):
    #filename = re.sub(r'\.tif$', '_dff_blurred_gauss_sigma=20.tif', video_files_mc[i])
    blurred = gaussian_filter(video, sigma=sigma)
    dff_blurred.append(blurred)
    #tff.imwrite(filename,blurred)


In [9]:
video_files_mc

['DIV27_CTRL-CTRL_W5_optostim_mc.tif',
 'DIV27_CTRL-CTRL_W4_optostim_mc.tif',
 'DIV27_CTRL-CTRL_W3_optostim_mc.tif',
 'DIV27_HS1-DA1M_W7_optostim_mc.tif',
 'DIV27_HS1-DA1M_W8_optostim_mc.tif',
 'DIV27_HS1-DA1M_W11_optostim_mc.tif',
 'DIV27_HS1-DA1M_W10_optostim_mc.tif',
 'DIV27_HS1-DA1M_W9_optostim_mc.tif']

In [10]:
wells = []

for filename in video_files_mc:
    match = re.search(r'W(\d+)', filename)
    if match:
        well = int(match.group(1))
        wells.append(well)

print(wells)

[5, 4, 3, 7, 8, 11, 10, 9]


In [13]:
#step 2 = low-pass filter the spatially blurred videos

from scipy.signal import butter, filtfilt

#BUG! so actual cutoff was 0.54 Hz not 0.5... fuck.
def butter_lowpass(cutoff, fps, order=5):
    nyq = 0.5 * fps #this line should be removed
    normal_cutoff = cutoff / nyq #this line should be removed
    return butter(order, normal_cutoff, btype='low', analog=False)

def butter_lowpass_filter(data, cutoff, fps, order=5):
    b, a = butter_lowpass(cutoff, fps, order=order)
    y = filtfilt(b, a, data)
    return y

order = 6     #order of the butterworth filter
cutoff = 0.45  # desired cutoff frequency of the filter, Hz; must be lower than 0.5 Hz due to one file having very low sampling rate...

all_global_trend = []

for i, video in enumerate(dff_blurred):
    filename = re.sub(r'\.tif$', '_dff_blurred_gauss_sigma=20_lp_cutoff=0pt45Hz.tif', video_files_mc[i])
    print(f"Generating {filename}")
    video_lp = np.zeros(video.shape, dtype=np.float32)
    if wells[i] == 7:
        fps = 0.939 # sample rate, Hz
    else:
        fps = 1.849 # sample rate, Hz
    for x in range(video.shape[1]):
        for y in range(video.shape[2]):
            video_lp[:,x,y] = butter_lowpass_filter(video[:,x,y], cutoff, fps, order)
    all_global_trend.append(video_lp)
    tff.imwrite(filename,video_lp)

Generating DIV27_CTRL-CTRL_W5_optostim_mc_dff_blurred_gauss_sigma=20_lp_cutoff=0pt45Hz.tif
Generating DIV27_CTRL-CTRL_W4_optostim_mc_dff_blurred_gauss_sigma=20_lp_cutoff=0pt45Hz.tif
Generating DIV27_CTRL-CTRL_W3_optostim_mc_dff_blurred_gauss_sigma=20_lp_cutoff=0pt45Hz.tif
Generating DIV27_HS1-DA1M_W7_optostim_mc_dff_blurred_gauss_sigma=20_lp_cutoff=0pt45Hz.tif
Generating DIV27_HS1-DA1M_W8_optostim_mc_dff_blurred_gauss_sigma=20_lp_cutoff=0pt45Hz.tif
Generating DIV27_HS1-DA1M_W11_optostim_mc_dff_blurred_gauss_sigma=20_lp_cutoff=0pt45Hz.tif
Generating DIV27_HS1-DA1M_W10_optostim_mc_dff_blurred_gauss_sigma=20_lp_cutoff=0pt45Hz.tif
Generating DIV27_HS1-DA1M_W9_optostim_mc_dff_blurred_gauss_sigma=20_lp_cutoff=0pt45Hz.tif


In [14]:
#step 3 = regress out the global trend videos from the original videos
#must process one video at a time (computationally intensive)

import sys,os
import tifffile as tff
import numpy as np
import re

output_dir = "/Users/cybellesmith/Box Sync/all/post_doc2/code/kiera_data_analysis/processed_videos"

os.chdir(output_dir)

video_files_mc = [
'DIV27_CTRL-CTRL_W5_optostim_mc.tif',
 'DIV27_CTRL-CTRL_W4_optostim_mc.tif',
 'DIV27_CTRL-CTRL_W3_optostim_mc.tif',
 'DIV27_HS1-DA1M_W7_optostim_mc.tif',
 'DIV27_HS1-DA1M_W8_optostim_mc.tif',
 'DIV27_HS1-DA1M_W11_optostim_mc.tif',
 'DIV27_HS1-DA1M_W10_optostim_mc.tif',
 'DIV27_HS1-DA1M_W9_optostim_mc.tif'  
]

for i, video_file in enumerate(video_files_mc):

    v_filename = re.sub(r'\.tif$', '_dff.tif', video_files_mc[i])
    video = tff.imread(v_filename)

    g_filename = re.sub(r'\.tif$', '_dff_blurred_gauss_sigma=20_lp_cutoff=0pt45Hz.tif', video_files_mc[i])
    g = tff.imread(g_filename)

    out_filename = re.sub(r'\.tif$', '_dff_resid.tif', video_files_mc[i])
    print(out_filename)
    
    video = video.astype(np.float32)
    g = g.astype(np.float32)
    
    g_centered = g - g.mean(axis=0)          # (T,H,W) minus (H,W)
    video_centered = video - video.mean(axis=0)
    
    den = (g_centered**2).sum(axis=0) + 1e-12    # (H,W)
    beta = (g_centered * video_centered).sum(axis=0) / den   # (H,W)
    resid = video_centered - g_centered * beta               # broadcasts (H,W) over time
    
    tff.imwrite(out_filename, resid.astype(np.float32))

DIV27_CTRL-CTRL_W5_optostim_mc_dff_resid.tif
DIV27_CTRL-CTRL_W4_optostim_mc_dff_resid.tif
DIV27_CTRL-CTRL_W3_optostim_mc_dff_resid.tif
DIV27_HS1-DA1M_W7_optostim_mc_dff_resid.tif
DIV27_HS1-DA1M_W8_optostim_mc_dff_resid.tif
DIV27_HS1-DA1M_W11_optostim_mc_dff_resid.tif
DIV27_HS1-DA1M_W10_optostim_mc_dff_resid.tif
DIV27_HS1-DA1M_W9_optostim_mc_dff_resid.tif


In [19]:
# #code to load in the residual files (full res)

# output_dir = "/Users/cybellesmith/Box Sync/all/post_doc2/code/kiera_data_analysis/processed_videos"

# video_files_mc = [
# 'DIV27_CTRL-CTRL_W5_optostim_mc.tif',
#  'DIV27_CTRL-CTRL_W4_optostim_mc.tif',
#  'DIV27_CTRL-CTRL_W3_optostim_mc.tif',
#  'DIV27_HS1-DA1M_W7_optostim_mc.tif',
#  'DIV27_HS1-DA1M_W8_optostim_mc.tif',
#  'DIV27_HS1-DA1M_W11_optostim_mc.tif',
#  'DIV27_HS1-DA1M_W10_optostim_mc.tif',
#  'DIV27_HS1-DA1M_W9_optostim_mc.tif'  
# ]

# os.chdir(output_dir)

# all_resid = []

# for i, video_file in enumerate(video_files_mc):

#     video_file = filename = re.sub(r'\.tif$', '_dff_resid.tif', video_files_mc[i])

#     print(video_file)

#     video = tff.imread(video_file)

#     all_resid.append(video)

DIV27_CTRL-CTRL_W5_optostim_mc_dff_resid.tif
DIV27_CTRL-CTRL_W4_optostim_mc_dff_resid.tif
DIV27_CTRL-CTRL_W3_optostim_mc_dff_resid.tif
DIV27_HS1-DA1M_W7_optostim_mc_dff_resid.tif
DIV27_HS1-DA1M_W8_optostim_mc_dff_resid.tif
DIV27_HS1-DA1M_W11_optostim_mc_dff_resid.tif
DIV27_HS1-DA1M_W10_optostim_mc_dff_resid.tif
DIV27_HS1-DA1M_W9_optostim_mc_dff_resid.tif
